# Codon-conditioned backbone signal: robustness analysis

This notebook stress-tests the codon-pair rejections shown in Figure 1. For every pair
flagged significant by at least one two-sample statistic, it asks: how many of the most
influential observations would need to be removed before the rejection disappears
(adversarial breakdown-k), and does the same rejection survive under label-randomized
null controls that preserve every observation's (phi, psi)?

Everything below is derived live from the same `cc-pvals.csv` files Figure 1 loads --
re-running the upstream pipeline and then re-running this notebook regenerates every
number here with no manual step.

In [1]:
import sys
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
sys.path.insert(0, str(Path("../../scripts/pnas2026").resolve()))

from _pnas2026_common import LABELLED_TAGS, compute_bh_rejections, load_all_pvals
from _common import SEED

from pp5.dihedral import flat_torus_distance
from pp5.distributions.kde import gaussian_kernel
from pp5.stats.breakdown import breakdown_k_kde, breakdown_k_mmd
from pp5.stats.controls import (
    gen_aa_ss_control_replicates,
    gen_pooled_shuffle_replicates,
    null_control_summary,
)

FDR = 0.05
K_PERM = 5000  # permutations for real-pair baseline p-values and breakdown-k checkpoints
K_CTRL = 2000  # permutations for control-replicate baseline/breakdown checkpoints
N_CONTROL_REPLICATES = 30

# torus-W2 breakdown-k uses a separate, R-backed algorithm and is slow (the last full
# run over 6 pairs took ~50 minutes). Set True to recompute it live; when False, values
# are loaded from TORUS_CACHE_PATH, with graceful "not computed" handling for any
# candidate pair missing from that file.
RUN_TORUS_BREAKDOWN_LIVE = False
TORUS_CACHE_PATH = Path("../../out/pnas-2026-phase1-golden/robustness_torus_summary.csv").resolve()

# The 5 statistics with an adversarial breakdown-k procedure.
BREAKDOWN_ARMS = [
    "kde-l1(bw=10.0)",
    "kde-l1(bw=CV)",
    "mmd(unbiased)",
    "mmd(biased)",
    "w2torus(4-fixed)",
]
FAST_ARMS = [arm for arm in BREAKDOWN_ARMS if arm != "w2torus(4-fixed)"]

OUT_DIR = Path("../../out/pnas-2026/figs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load p-values

Same registry and loader Figure 1 uses (`_pnas2026_common.py`), so this notebook can never
diverge on which runs or thresholds are in play.

In [2]:
df_all_pvals = load_all_pvals(LABELLED_TAGS)
df_rejections = compute_bh_rejections(df_all_pvals, fdr=FDR)
real_rejections = df_rejections[
    (df_rejections["codon_randomization"] == "none")
    & (df_rejections["stat_test"].isin(BREAKDOWN_ARMS))
].reset_index(drop=True)
real_rejections

,stat_test,codon_randomization,SS,n_hypotheses,fdr,bh_pvalue_threshold,n_rejected,rejected_pairs
0,w2torus(4-fixed),none,HELIX,87,0.05,0.002299,4,"[L-CTC:L-TTG, L-CTC:L-CTG, L-CTC:L-CTT, R-AGG:..."
1,w2torus(4-fixed),none,OTHER,87,0.05,0.000000,0,[]
2,w2torus(4-fixed),none,SHEET,87,0.05,0.000000,0,[]
3,w2torus(4-fixed),none,TURN,87,0.05,0.000575,1,[A-GCG:A-GCT]
4,kde-l1(bw=10.0),none,HELIX,87,0.05,0.001149,2,"[L-CTC:L-TTG, L-CTC:L-CTG]"
5,kde-l1(bw=10.0),none,OTHER,87,0.05,0.000000,0,[]
6,kde-l1(bw=10.0),none,SHEET,87,0.05,0.000000,0,[]
7,kde-l1(bw=10.0),none,TURN,87,0.05,0.001149,2,"[P-CCC:P-CCG, A-GCG:A-GCT]"
8,kde-l1(bw=CV),none,HELIX,87,0.05,0.000575,1,[L-CTC:L-TTG]
9,kde-l1(bw=CV),none,OTHER,87,0.05,0.000000,0,[]


## Load raw inputs

The pvals CSVs only carry aggregated statistics per codon pair; the breakdown-k and
provenance analysis below needs the underlying per-position data: the aggregated
dataset, the cross-validated KDE bandwidths (for the KDE-CV arm), and the raw
per-structure records (for PDB provenance).

In [3]:
DATASET_PATH = Path(
    "../../out/prec-collected/20211001_124553-aida-ex_EC-src_EC/results/"
    "pointwise_cdist-natcom/_intermediate_/dataset.csv"
).resolve()
CV_BANDWIDTH_PATH = Path("../../out/pnas-2026/bandwidth_cv/kernel_bandwidths.csv").resolve()
PROVENANCE_PATH = Path(
    "../../out/prec-collected/20211001_124553-aida-ex_EC-src_EC/data-precs.csv"
).resolve()

assert DATASET_PATH.is_file(), DATASET_PATH
assert CV_BANDWIDTH_PATH.is_file(), CV_BANDWIDTH_PATH
assert PROVENANCE_PATH.is_file(), PROVENANCE_PATH

df_dataset = pd.read_csv(DATASET_PATH)
if "AA" not in df_dataset.columns:
    df_dataset["AA"] = df_dataset["codon"].str.split("-").str[0]

df_cv_bandwidth = pd.read_csv(CV_BANDWIDTH_PATH).set_index(["codon", "ss"])["sigma_cv_deg"]

## Determine candidate pairs

A pair is a candidate for breakdown-k if at least one of the 5 statistics above rejects
it in the real data, at that statistic's own BH threshold. This replaces any fixed pair
list: a future rerun with different p-values changes the candidate set automatically.

In [4]:
candidate_pairs = sorted(
    {
        (row.SS, pair)
        for row in real_rejections.itertuples()
        for pair in row.rejected_pairs
    }
)
print(f"{len(candidate_pairs)} candidate pairs:")
for ss, pair in candidate_pairs:
    print(f"  {ss}  {pair}")

# Sanity check against the historical stress-test set documented in the revision plan
# (Section 4): the same union should still appear, since these pvals are the ones that
# set produced.
HISTORICAL_UNION = {
    ("HELIX", "L-CTC:L-TTG"),
    ("HELIX", "L-CTC:L-CTG"),
    ("HELIX", "L-CTC:L-CTT"),
    ("HELIX", "R-AGG:R-CGA"),
    ("TURN", "A-GCG:A-GCT"),
    ("TURN", "P-CCC:P-CCG"),
}
missing = HISTORICAL_UNION - set(candidate_pairs)
assert not missing, f"Historical stress-test pairs missing from the candidate set: {missing}"

6 candidate pairs:
  HELIX  L-CTC:L-CTG
  HELIX  L-CTC:L-CTT
  HELIX  L-CTC:L-TTG
  HELIX  R-AGG:R-CGA
  TURN  A-GCG:A-GCT
  TURN  P-CCC:P-CCG


## Shared helpers

`codon_angles` extracts one codon's (phi, psi) points (in radians) for a secondary-structure
class, plus the (unp_id, unp_idx) key of each point (needed later to trace adversarially-removed
points back to PDB structures). `k_grid_for` is the adversarial-removal checkpoint schedule,
used identically by every arm below.

In [5]:
def codon_angles(df: pd.DataFrame, ss: str, codon: str) -> tuple[np.ndarray, np.ndarray]:
    """Radian (phi, psi) points and (unp_id, unp_idx) keys for one (SS, codon) group.

    :param df: Dataset with columns `condition_group`, `codon`, `phi`, `psi`, `unp_id`,
        `unp_idx`.
    :param ss: Secondary-structure class, e.g. `"HELIX"`.
    :param codon: Codon label, e.g. `"L-CTC"`.
    :return: `(n, 2)` radian phi/psi array, `(n, 2)` object array of (unp_id, unp_idx).
    """
    sub = df[(df["condition_group"] == ss) & (df["codon"] == codon)]
    points = np.deg2rad(sub[["phi", "psi"]].to_numpy())
    keys = sub[["unp_id", "unp_idx"]].to_numpy()
    return points, keys


def k_grid_for(n_min: int) -> list[int]:
    """Adaptive breakdown-k checkpoint grid, capped so the smaller group keeps >= 2 points.

    :param n_min: Size of the smaller of the two groups being compared.
    :return: Sorted list of checkpoint values, each `<= n_min - 2`.
    """
    k_cap = min(40, max(5, int(np.ceil(0.05 * n_min))))
    grid = sorted({0, 1, 2, 3, 5, 8, 12, 20, k_cap})
    return [k for k in grid if k <= n_min - 2]